# TRINETRA — DistilBERT Crypto Classifier v2

Fine-tunes `distilbert-base-uncased` on the dataset built by
`backend/scripts/build_training_dataset.py`.

**Before running:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Upload `train.jsonl`, `val.jsonl`, `test.jsonl` to the file panel (left sidebar)
3. Run all cells

The label schema matches `loaded_model/crypto_classifier/config.json` exactly:
`0 QUANTUM_VULNERABLE`, `1 CLASSICAL_SAFE`, `2 PQC_READY`, `3 CLEAN`.

Expected training time: **30–45 minutes**.

## Cell 1 — Setup

In [ ]:
# Run on Google Colab with T4 GPU
# Runtime -> Change runtime type -> GPU (T4)
# Upload train.jsonl, val.jsonl, test.jsonl to the file panel
# Expected training time: 30-45 minutes

!pip install transformers datasets evaluate scikit-learn torch accelerate -q

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected - training will be very slow.')
    print('Runtime -> Change runtime type -> GPU (T4)')

## Cell 2 — Load data

In [ ]:
from collections import Counter

from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train': 'train.jsonl',
    'validation': 'val.jsonl',
    'test': 'test.jsonl'
})

print(dataset)

LABEL_NAMES = ['QUANTUM_VULNERABLE', 'CLASSICAL_SAFE', 'PQC_READY', 'CLEAN']
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}

for split in dataset:
    counts = Counter(dataset[split]['label'])
    n = len(dataset[split])
    print(f'\n{split} (n={n})')
    for name in LABEL_NAMES:
        c = counts.get(name, 0)
        print(f'  {name:<20} {c:>5}  ({c / n * 100:>5.1f}%)')

# Any class the split is missing will not be learnable -- surface it now.
missing = [n for n in LABEL_NAMES if n not in set(dataset['train']['label'])]
if missing:
    print('\nWARNING: classes absent from train:', missing)

## Cell 3 — Tokenize

In [ ]:
from transformers import DistilBertTokenizerFast

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 512

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


def tokenize(batch):
    encoded = tokenizer(
        batch['text'],
        max_length=MAX_LENGTH,
        padding=True,
        truncation=True,
    )
    encoded['labels'] = [label2id[l] for l in batch['label']]
    return encoded


tokenized = dataset.map(tokenize, batched=True, remove_columns=['text', 'label'])
print(tokenized)
print('\nfirst train example keys:', list(tokenized['train'][0].keys()))

## Cell 4 — Model setup

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=4,
    id2label={0: 'QUANTUM_VULNERABLE', 1: 'CLASSICAL_SAFE', 2: 'PQC_READY', 3: 'CLEAN'},
    label2id={'QUANTUM_VULNERABLE': 0, 'CLASSICAL_SAFE': 1, 'PQC_READY': 2, 'CLEAN': 3},
)

print('labels:', model.config.id2label)
print('params:', f'{model.num_parameters():,}')

## Cell 5 — Training

In [ ]:
import inspect

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer, TrainingArguments


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    _, _, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'macro_f1': macro_f1,
    }


args_kwargs = dict(
    output_dir='./results',
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=25,
    report_to='none',
)

# transformers renamed evaluation_strategy -> eval_strategy in 4.46.
# Pick whichever this runtime's TrainingArguments actually accepts.
_params = inspect.signature(TrainingArguments.__init__).parameters
args_kwargs['eval_strategy' if 'eval_strategy' in _params else 'evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)

## Cell 6 — Evaluate on the test set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(tokenized['test'])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

metrics = compute_metrics((predictions.predictions, labels))
print('=' * 58)
print('TEST SET RESULTS')
print('=' * 58)
print(f"accuracy    : {metrics['accuracy']:.4f}")
print(f"weighted F1 : {metrics['f1']:.4f}")
print(f"macro F1    : {metrics['macro_f1']:.4f}")

present = sorted(set(labels.tolist()) | set(preds.tolist()))
target_names = [id2label[i] for i in present]

print('\nPer-class metrics:')
print(classification_report(labels, preds, labels=present,
                            target_names=target_names, zero_division=0, digits=4))

cm = confusion_matrix(labels, preds, labels=present)
width = max(len(n) for n in target_names) + 2
print('Confusion matrix (rows = actual, cols = predicted):\n')
print(' ' * width + ''.join(f'{n[:10]:>12}' for n in target_names))
for name, row in zip(target_names, cm):
    print(f'{name:<{width}}' + ''.join(f'{v:>12}' for v in row))

per_class_f1 = {}
from sklearn.metrics import f1_score
f1s = f1_score(labels, preds, labels=present, average=None, zero_division=0)
for i, score in zip(present, f1s):
    per_class_f1[id2label[i]] = score
print('\nper-class F1:', {k: round(float(v), 4) for k, v in per_class_f1.items()})

## Cell 7 — Save the model

Download the whole `distilbert-trinetra-crypto-v2/` folder and copy its contents into
`backend/engine/ai/loaded_model/crypto_classifier_v2/` in the repo.
`settings.ai_model_dir` already points there.

In [ ]:
trainer.save_model('./distilbert-trinetra-crypto-v2')
tokenizer.save_pretrained('./distilbert-trinetra-crypto-v2')

import os
print('saved files:')
for f in sorted(os.listdir('./distilbert-trinetra-crypto-v2')):
    size = os.path.getsize(os.path.join('./distilbert-trinetra-crypto-v2', f))
    print(f'  {f:<32} {size / 1024:>9.1f} KB')

# Zip it for a single download
!zip -qr distilbert-trinetra-crypto-v2.zip distilbert-trinetra-crypto-v2
print('\nzip ready: distilbert-trinetra-crypto-v2.zip')

## Cell 8 — Summary block (copy-paste)

In [ ]:
best_epoch = None
for entry in trainer.state.log_history:
    if 'eval_f1' in entry and trainer.state.best_metric is not None:
        if abs(entry['eval_f1'] - trainer.state.best_metric) < 1e-9:
            best_epoch = entry.get('epoch')
            break


def f1_of(name):
    value = per_class_f1.get(name)
    return f'{value:.4f}' if value is not None else 'n/a'


print('===== TRINETRA DistilBERT Crypto Classifier v2 =====')
print(f"Training samples: {len(tokenized['train'])}")
print(f"Validation samples: {len(tokenized['validation'])}")
print(f"Test samples: {len(tokenized['test'])}")
print('Epochs: 5')
print(f"Best epoch: {best_epoch if best_epoch is not None else 'n/a'}")
print(f"Weighted F1: {metrics['f1']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print('Per-class F1: '
      f"QV={f1_of('QUANTUM_VULNERABLE')}, "
      f"CS={f1_of('CLASSICAL_SAFE')}, "
      f"PR={f1_of('PQC_READY')}, "
      f"CL={f1_of('CLEAN')}")
print('=================================================')

## Cell 9 — Optional: push to HuggingFace Hub

In [ ]:
# Optional. Uncomment and set your username to publish the model.
#
# from huggingface_hub import notebook_login
# notebook_login()
#
# HF_USERNAME = '<your-username>'   # <-- set this
# REPO = f'{HF_USERNAME}/distilbert-trinetra-crypto-v2'
#
# model.push_to_hub(REPO, private=True)
# tokenizer.push_to_hub(REPO, private=True)
# print('pushed to', REPO)